# COSC 2060: Modern Data Engineering & AI Infrastructure
## Unit 08: Re-Platforming Capstone Production Defense (Fragile Local Scripts → Docker & Defensive Pytest Gates)

---

### 🏢 The Workplace Scenario
Your senior data science colleague, **Dr. Anton Weber**, developed the end-to-end logistics analytics engine. He packaged his deliverable as a loose collection of Python scripts with hardcoded local paths (`C:\Users\anton\Desktop\data...`), unpinned library versions, and zero automated tests. When asked whether the pipeline was ready for the production cloud deployment, Dr. Weber famously replied: *"It works on my machine."*

When the infrastructure engineering team attempted to run his script inside an automated Linux container on a blind test batch with real-world edge cases (corrupted negative fuel readings, fuel theft spikes, and emergency recalls), the script crashed immediately with `FileNotFoundError`, produced unhandled exceptions, and silently allowed invalid records through.

### 🎯 Your Mission as AI Data Engineer
1. **Analyze Dr. Weber's Fragile Prototype**: Understand how hardcoded local paths, unpinned dependencies, and lack of error handling cause production deployment failure.
2. **Expose the Blind Batch Crashes**: Run Dr. Weber's prototype against an evaluation batch containing corrupt data and observe the crash.
3. **Re-Platform into a Containerized Production Package**:
   - Author a production-grade **Dockerfile** with locked dependencies and non-root execution.
   - Implement the `CapstoneDefenseSystem` with **Dead-Letter Queue (DLQ)** isolation, fuel fraud detection, and multimodal safety clearance.
4. **Assert Production Quality Gates**: Write and execute a comprehensive **Pytest** test suite verifying that corrupt records are diverted to the DLQ, fraud alerts are triggered, and compromised trucks are grounded.
5. **Verify Blind Regression Pass**: Confirm 100% test pass rate across the blind validation suite.

In [ ]:
import time
import json
import os
import pytest
from typing import Dict, List, Any

print("[*] Loading blind evaluation batch for Capstone Defense...")

# Blind batch containing real-world dirty data & edge cases
BLIND_EVAL_DATA = {
    "fuel_transactions": [
        {"tx_id": "TX-901", "truck_id": "TRUCK-101", "gallons": 140.0, "state": "OH"},
        {"tx_id": "TX-902", "truck_id": "TRUCK-219", "gallons": 340.0, "state": "IN"}, # Fraud spike (>300 gal)
        {"tx_id": "TX-903", "truck_id": "TRUCK-304", "gallons": -15.0, "state": "IL"}, # Corrupt negative value!
        {"tx_id": "TX-904", "truck_id": "TRUCK-405", "gallons": 95.0,  "state": "OH"}
    ],
    "vision_inspections": [
        {"truck_id": "TRUCK-219", "defect": "sidewall_crack", "confidence": 0.94},
        {"truck_id": "TRUCK-101", "defect": "none", "confidence": 0.99}
    ],
    "maintenance_logs": [
        {"truck_id": "TRUCK-219", "wo": "WO-9901", "notes": "Engine whistling noise. Active recall notice Part #TC-884 on turbo."},
        {"truck_id": "TRUCK-101", "wo": "WO-9902", "notes": "Wiper blade replacement completed."}
    ]
}

print(f"[+] Loaded {len(BLIND_EVAL_DATA['fuel_transactions'])} transactions and {len(BLIND_EVAL_DATA['vision_inspections'])} inspections.")

In [ ]:
# ==============================================================================
# TIER 1: The Data Science Prototype (Dr. Anton Weber)
# ==============================================================================
# Dr. Weber's script assumes hardcoded local file paths and clean numbers.

def run_anton_prototype(transactions, local_path_check=True):
    # Fragile assumption 1: Hardcoded developer machine path
    dev_path = "C:\\Users\\anton\\Desktop\\capstone_data"
    if local_path_check and not os.path.exists(dev_path):
        raise FileNotFoundError(f"Missing local directory: {dev_path}. Cannot find weights!")
        
    # Fragile assumption 2: All numbers are positive and valid
    total_gal = 0.0
    for tx in transactions:
        total_gal += tx["gallons"] # Corrupt negative values silently corrupt the sum!
        
    return {"total_fleet_fuel": total_gal, "status": "COMPLETED"}

print("[*] Testing Dr. Weber's prototype logic on clean inputs...")
# Simulate running on his laptop where the path exists
anton_output = run_anton_prototype(BLIND_EVAL_DATA["fuel_transactions"][:2], local_path_check=False)
print(f"[+] Output on Local Laptop: {anton_output}")

### ⚠️ The Deployment & Quality Bottleneck

Why does Dr. Weber's script fail when handed off to the enterprise?
1. **Environment Drift ("Works on My Machine")**: Hardcoded file paths and unpinned dependencies cause immediate crashes when run inside a Linux Docker container or CI/CD test runner.
2. **Corrupted Data Poisoning**: The script blindly adds `-15.0` gallons to fleet totals without sanity checks, corrupting the feature store.
3. **No Safety Interlocks**: Fuel fraud spikes (340 gallons in a 250-gallon tank) and critical vision tire cracks pass through without alerts.

Let's witness these production failures.

In [ ]:
# ==============================================================================
# TIER 2: Deployment & Data Corruption Failure
# ==============================================================================
# Failure A: Hardcoded path in foreign environment
print("[*] Attempting deployment on production Linux container...")
try:
    run_anton_prototype(BLIND_EVAL_DATA["fuel_transactions"], local_path_check=True)
except FileNotFoundError as e:
    print(f"[!] DEPLOYMENT CRASH: {e}")

# Failure B: Silent data corruption
dirty_run = run_anton_prototype(BLIND_EVAL_DATA["fuel_transactions"], local_path_check=False)
print(f"[!] SILENT DATA CORRUPTION: Total gallons calculated as {dirty_run['total_fleet_fuel']}.")
print("    --> Negative 15.0 gallons was subtracted from fuel totals without any warning or DLQ!")

### 🚀 The Data Engineering Solution: Defensive Packaging & Docker

To deliver production-grade defense, we re-platform into an immutable containerized architecture:
1. **Standardized Docker Container**: A container specification with pinned dependencies, environment variables (`DATA_DIR=/app/data`), and multi-stage building.
2. **Defensive Schema Ingestion & Dead-Letter Queue (DLQ)**: Corrupted records (`gallons <= 0`) are safely quarantined into a DLQ for human review without crashing the ingestion worker.
3. **Fraud & Safety Anomaly Interlocks**: Business invariants (`gallons > 300`) trigger instant telemetry alerts, and multimodal defect signals ground high-risk vehicles.
4. **Automated Pytest Regression Suite**: Every invariant is verified via automated assertions before deployment.

In [ ]:
# ==============================================================================
# TIER 3: The Data Engineering Refactor (Production Defensive Architecture)
# ==============================================================================
class ProductionDefenseEngine:
    def __init__(self, service_name: str = "Enterprise-Logistics-Core"):
        self.service_name = service_name
        self.feature_store: Dict[str, Dict[str, float]] = {}
        self.dead_letter_queue: List[Dict[str, Any]] = []
        self.active_alerts: List[Dict[str, Any]] = []
        self.grounded_trucks: set = set()

    def ingest_fuel_stream(self, transactions: List[Dict[str, Any]]):
        for tx in transactions:
            # Defensive Gate 1: Quarantine invalid negative/zero fuel to DLQ
            if tx.get("gallons", 0) <= 0.0:
                self.dead_letter_queue.append({
                    "record": tx,
                    "reason": "Invalid non-positive fuel measurement (gallons <= 0)",
                    "timestamp": time.time()
                })
                continue
                
            # Defensive Gate 2: Flag fuel fraud anomalies (> 300 gal)
            if tx["gallons"] > 300.0:
                self.active_alerts.append({
                    "alert_type": "FUEL_FRAUD_ALERT",
                    "truck_id": tx["truck_id"],
                    "detail": f"Transaction {tx['tx_id']}: {tx['gallons']} gal exceeds maximum tank capacity."
                })
                
            # Ingestion into feature store
            tid = tx["truck_id"]
            if tid not in self.feature_store:
                self.feature_store[tid] = {"total_gallons": 0.0, "tx_count": 0}
            self.feature_store[tid]["total_gallons"] += tx["gallons"]
            self.feature_store[tid]["tx_count"] += 1

    def evaluate_fleet_safety(self, truck_id: str, vision_data: List[Dict], rag_data: List[Dict]) -> Dict[str, Any]:
        truck_defects = [v for v in vision_data if v["truck_id"] == truck_id]
        truck_rag = [r for r in rag_data if r["truck_id"] == truck_id]
        
        has_tire_crack = any(d.get("defect") == "sidewall_crack" for d in truck_defects)
        has_recall = any("recall" in r.get("notes", "").lower() for r in truck_rag)
        
        if has_tire_crack or has_recall:
            self.grounded_trucks.add(truck_id)
            reasons = []
            if has_tire_crack:
                reasons.append("Computer Vision: Sidewall stress fracture.")
            if has_recall:
                reasons.append("Maintenance RAG: Active safety recall.")
            return {"truck_id": truck_id, "clearance": "GROUNDED", "reasons": reasons}
        else:
            return {"truck_id": truck_id, "clearance": "CLEARED", "reasons": ["All safety checks passed."]}

print("[+] ProductionDefenseEngine initialized.")

# Companion Dockerfile Specification (Standard OCI Container)
dockerfile_content = """# Production Container Specification: COSC 2060 Capstone
FROM python:3.11-slim
WORKDIR /app
COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt
COPY . .
ENV DATA_DIR=/app/data
USER nobody
CMD ["pytest", "-v", "tests/"]
"""
print(f"[+] Dockerfile Specification:\n{dockerfile_content.strip()}")

In [ ]:
# ==============================================================================
# QUALITY GATE: Defensive Contract & Blind Regression Verification
# ==============================================================================
engine = ProductionDefenseEngine()

print("[*] Executing defensive pipeline against blind evaluation batch...")
engine.ingest_fuel_stream(BLIND_EVAL_DATA["fuel_transactions"])

# Test 1: Verify Dead-Letter Queue (Corrupt record TX-903 must be quarantined)
assert len(engine.dead_letter_queue) == 1, f"Expected 1 DLQ record, got: {len(engine.dead_letter_queue)}"
assert engine.dead_letter_queue[0]["record"]["tx_id"] == "TX-903"
print("[+] DLQ Verification: PASS (Corrupt record TX-903 safely isolated).")

# Test 2: Verify Fuel Fraud Detection (Transaction TX-902 must trigger alert)
assert len(engine.active_alerts) == 1, f"Expected 1 fraud alert, got: {len(engine.active_alerts)}"
assert engine.active_alerts[0]["alert_type"] == "FUEL_FRAUD_ALERT"
assert engine.active_alerts[0]["truck_id"] == "TRUCK-219"
print("[+] Fraud Alert Verification: PASS (TX-902 340-gallon fraud alert fired).")

# Test 3: Verify Feature Store Cleanliness (Negative fuel was NOT added)
total_clean_fuel = sum(v["total_gallons"] for v in engine.feature_store.values())
expected_clean = 140.0 + 340.0 + 95.0 # 575.0 gallons
assert total_clean_fuel == expected_clean, f"Feature store contaminated: {total_clean_fuel} vs {expected_clean}"
print(f"[+] Feature Store Integrity: PASS (Clean Total: {total_clean_fuel} gal).")

# Test 4: Verify Multimodal Grounding on TRUCK-219 (Tire crack + recall)
eval_219 = engine.evaluate_fleet_safety("TRUCK-219", BLIND_EVAL_DATA["vision_inspections"], BLIND_EVAL_DATA["maintenance_logs"])
assert eval_219["clearance"] == "GROUNDED"
assert len(eval_219["reasons"]) == 2
print("[+] Safety Grounding Verification: PASS (TRUCK-219 grounded for 2 violations).")

# Test 5: Verify Operational Clearance on TRUCK-101
eval_101 = engine.evaluate_fleet_safety("TRUCK-101", BLIND_EVAL_DATA["vision_inspections"], BLIND_EVAL_DATA["maintenance_logs"])
assert eval_101["clearance"] == "CLEARED"
print("[+] Operational Clearance Verification: PASS (TRUCK-101 cleared).")

print("=" * 60)
print("[+] PASS: 100% DEFENSIVE CONTRACT QUALITY GATE VERIFIED!")
print("    All 5 blind regression assertions passed with zero unhandled exceptions.")
print("=" * 60)

In [ ]:
# ==============================================================================
# PERFORMANCE & RESILIENCE AUDIT: Summary Scorecard
# ==============================================================================
print("-" * 65)
print(f"{'Defensive Gate':<30} | {'Prototype':<15} | {'Production Engine'}")
print("-" * 65)
print(f"{'Environment Portability':<30} | {'Hardcoded Path':<15} | {'Docker OCI Container'}")
print(f"{'Corrupted Data Handling':<30} | {'Silent Drift':<15} | {'Dead-Letter Queue (DLQ)'}")
print(f"{'Anomaly & Fraud Defense':<30} | {'Ignored':<15} | {'Automated Alert Rules'}")
print(f"{'Multimodal Safety Interlock':<30} | {'Unchecked':<15} | {'Multi-Factor Grounding'}")
print(f"{'Regression Testing':<30} | {'Manual Hope':<15} | {'Automated Pytest Suite'}")
print("-" * 65)
print("[+] Capstone Defense Scorecard: 100% PASS.")

### 🎓 Key Takeaways for AI Data Engineers

1. **Eliminate 'Works on My Machine'**: Containerize applications using Docker so code, dependencies, and environment variables are identical across development, CI/CD, and production.
2. **Defensive Ingestion via Dead-Letter Queues (DLQ)**: In production data engineering, dirty data is inevitable. Never let bad data crash your pipeline or contaminate analytical feature stores; route it to a DLQ for alerting and replay.
3. **Automate Invariants with Tests**: Pytest is not just for software developers. Data engineers use automated assertions to guarantee data contracts and prevent catastrophic model drift.
4. **Bridge to Unit 08 Applied Capstone Defense**: You are now fully equipped to present and defend your complete 8-unit production data engineering ecosystem in the Unit 08 Live Capstone Defense!